In [ ]:
###############################################################################
# IMPORTS
###############################################################################

import os
import sys
import time
import glob
import math
import pickle
import argparse
import warnings
from collections import defaultdict
from typing import Dict, List, Literal, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import dimod
from dimod import BinaryQuadraticModel, make_quadratic, BINARY
from dwave.system import DWaveSampler, EmbeddingComposite, FixedEmbeddingComposite
import minorminer
import dwave_networkx as dnx
from neal import SimulatedAnnealingSampler

In [ ]:
sampler_quantum = DWaveSampler(solver='Advantage2_system1', token='<INPUT D-WAVE TOKEN HERE>')
sampler_classic = SimulatedAnnealingSampler()

In [ ]:
###############################################################################
# BROOKS' THEOREM BOUND
###############################################################################

def brooks_bound(G: nx.Graph) -> int:
    
    """
    Upper bound on chromatic number via Brooks' theorem.
    For connected G that is neither complete nor an odd cycle: χ(G) ≤ Δ(G).
    For complete graphs K_n:  χ = n = Δ + 1.
    For odd cycles C_{2k+1}: χ = 3 = Δ + 1.
    For disconnected graphs:  max over components.
    """
    
    if G.number_of_nodes() == 0:
        return 0
    if G.number_of_nodes() == 1:
        return 1
    if G.number_of_edges() == 0:
        return 1

    max_chi = 1
    for comp in nx.connected_components(G):
        H = G.subgraph(comp).copy()
        n = H.number_of_nodes()
        delta = max(d for _, d in H.degree()) if n > 0 else 0

        if delta <= 1:
            chi = min(2, n)
        elif delta == 2:
            # Δ=2 → path or cycle.  Odd cycle needs 3 colors.
            if H.number_of_edges() == n and n % 2 == 1:
                chi = 3
            else:
                chi = 2
        else:
            # Complete graph check: |E| = n(n-1)/2
            if H.number_of_edges() == n * (n - 1) // 2:
                chi = n
            else:
                chi = delta
        max_chi = max(max_chi, chi)

    return max_chi

In [ ]:
###############################################################################
# ONE-HOT ENCODING
###############################################################################

def build_onehot_qubo(G: nx.Graph, C: int):
    
    """
    One-hot QUBO for Minimum Graph Coloring.

    Variables:
    ---------
    x_{v,c} : 1 iff vertex v gets color c     (|V|·C  variables)
    y_c     : 1 iff color c is used           (C       variables)

    Hamiltonian:
    -----------
    H = A1 Σ_v (1 - Σ_c x_{v,c})²            one-hot
      + A2 Σ_{(u,v)∈E} Σ_c x_{u,c} x_{v,c}   adjacency
      + A3 Σ_c y_c                           minimize colors
      + A4 Σ_v Σ_c x_{v,c}(1 - y_c)          link

    Penalties:  A3=1,  A4=|V|+1,  A2=C·A4+1,  A1=(|E|+1)·A2
    Logical qubits: C·(|V|+1)

    Returns (bqm, info_dict).
    """
    
    nodes = sorted(G.nodes())
    edges = list(G.edges())
    n, m  = len(nodes), len(edges)
    nid   = {v: i for i, v in enumerate(nodes)}

    A3 = 1.0
    A4 = float(n + 1)
    A2 = C * A4 + 1.0
    A1 = (m + 1) * A2

    xv = lambda v, c: f"x_{nid[v]}_{c}"
    yc = lambda c: f"y_{c}"

    bqm = BinaryQuadraticModel(vartype=BINARY)

    # H1: one-hot   A1·Σ_v [ 1 - Σ_c x_{v,c} + 2·Σ_{c<c'} x_{v,c} x_{v,c'} ]
    for v in nodes:
        for c in range(C):
            bqm.add_linear(xv(v, c), -A1)
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                bqm.add_quadratic(xv(v, c1), xv(v, c2), 2.0 * A1)

    # H2: adjacency
    for u, v in edges:
        for c in range(C):
            bqm.add_quadratic(xv(u, c), xv(v, c), A2)

    # H3: minimize colors
    for c in range(C):
        bqm.add_linear(yc(c), A3)

    # H4: link  A4·[ x_{v,c} - x_{v,c} y_c ]
    for v in nodes:
        for c in range(C):
            bqm.add_linear(xv(v, c), A4)
            bqm.add_quadratic(xv(v, c), yc(c), -A4)

    bqm.offset += A1 * n  # constant from (1 - ...)^2

    info = {
        "encoding": "one-hot",
        "num_colors": C,
        "num_logical_qubits": C * (n + 1),
        "num_logical_qubits_pre_quad": C * (n + 1),
        "num_auxiliary": 0,
        "penalties": {"A1": A1, "A2": A2, "A3": A3, "A4": A4},
    }
    return bqm, info


def decode_onehot(sample: dict, G: nx.Graph, C: int):

    """Decode one-hot sample → {vertex: color} or None if constraint violated."""
    
    nodes = sorted(G.nodes())
    nid = {v: i for i, v in enumerate(nodes)}
    coloring = {}
    for v in nodes:
        colors = [c for c in range(C) if sample.get(f"x_{nid[v]}_{c}", 0) == 1]
        if len(colors) != 1:
            return None
        coloring[v] = colors[0]
    return coloring

In [ ]:
###############################################################################
# BINARY (LOGARITHMIC) ENCODING
###############################################################################

def _poly_mul(p1: dict, p2: dict) -> dict:
    
    """
    Multiply two multilinear polynomials over binary variables.
    Represented as  {frozenset_of_vars: coeff}.
    Variables from different bit-positions never collide.
    """
    
    result = defaultdict(float)
    for v1, c1 in p1.items():
        for v2, c2 in p2.items():
            result[v1 | v2] += c1 * c2
    return dict(result)


def _expand_edge_product(ui: int, vi: int, L: int) -> dict:
    
    """
    Expand  Π_{k=0}^{L-1} (1 - b_{u,k} - b_{v,k} + 2·b_{u,k}·b_{v,k})
    = Π_k XNOR(b_{u,k}, b_{v,k}).

    Returns {frozenset((vertex_idx, bit_idx), ...): coeff}.
    """
    
    poly = {frozenset(): 1.0}
    for k in range(L):
        factor = {
            frozenset():                  1.0,
            frozenset({(ui, k)}):        -1.0,
            frozenset({(vi, k)}):        -1.0,
            frozenset({(ui, k), (vi, k)}): 2.0,
        }
        poly = _poly_mul(poly, factor)
    return {k: v for k, v in poly.items() if abs(v) > 1e-15}


def build_binary_hubo(G: nx.Graph, C: int):
    
    """
    Binary HUBO for Minimum Graph Coloring:

        H(b) = A · Σ_{(u,v)∈E} Π_k XNOR(b_{u,k}, b_{v,k})
             + Σ_k P_k · Σ_v b_{v,k}

    Penalties:
        P_k = (|V|+1)^{k-1}     for k = 1, …, L
        A   = |V| · Σ_k P_k + 1

    L = ⌈log₂ C⌉  bits per vertex.   Pre-quad logical qubits = |V|·L.

    Returns (hubo_dict, info_dict).
    hubo_dict: {tuple_of_var_names: coeff}  (dimod.make_quadratic format).
    """
    
    nodes = sorted(G.nodes())
    edges = list(G.edges())
    n = len(nodes)
    nid = {v: i for i, v in enumerate(nodes)}

    L = max(1, math.ceil(math.log2(C))) if C > 1 else 1

    # Penalties
    Pk = [(n + 1) ** k for k in range(L)]        # P_1=1, P_2=n+1, …
    A  = float(n * sum(Pk) + 1)

    bvar = lambda vi, k: f"b_{vi}_{k}"

    hubo = defaultdict(float)

    # Adjacency term
    for u, v in edges:
        poly = _expand_edge_product(nid[u], nid[v], L)
        for fset, coeff in poly.items():
            key = tuple(sorted(bvar(vi, ki) for vi, ki in fset))
            hubo[key] += A * coeff

    # Lexicographic penalty
    for k in range(L):
        for v in nodes:
            key = (bvar(nid[v], k),)
            hubo[key] += Pk[k]

    offset = hubo.pop((), 0.0)
    hubo = {k: v for k, v in hubo.items() if abs(v) > 1e-15}

    max_deg = max((len(k) for k in hubo), default=0)

    info = {
        "encoding": "binary",
        "num_colors": C,
        "L": L,
        "num_logical_qubits_pre_quad": n * L,
        "hubo_max_degree": max_deg,
        "hubo_num_terms": len(hubo),
        "offset": offset,
        "penalties": {"A": A, "Pk": Pk},
    }
    return dict(hubo), info


def quadratize_binary(hubo: dict, info: dict, G: nx.Graph):
    
    """
    Quadratize via dimod.make_quadratic.

    Penalty strength:  M = A + |V|·Σ_k P_k + 1.

    Returns (bqm, updated_info).
    """
    
    n  = G.number_of_nodes()
    A  = info["penalties"]["A"]
    Pk = info["penalties"]["Pk"]
    M  = A + n * sum(Pk) + 1.0

    bqm = make_quadratic(hubo, strength=M, vartype=BINARY)
    bqm.offset += info.get("offset", 0.0)

    n_pre  = info["num_logical_qubits_pre_quad"]
    n_post = bqm.num_variables
    n_aux  = n_post - n_pre

    L = info["L"]
    theoretical_aux = G.number_of_edges() * max(0, 2 * L - 2)

    info.update({
        "num_logical_qubits_post_quad": n_post,
        "num_auxiliary": n_aux,
        "theoretical_auxiliary": theoretical_aux,
        "quad_strength": M,
    })
    return bqm, info


def decode_binary(sample: dict, G: nx.Graph, L: int) -> dict:
    
    """Decode binary sample → {vertex: color}."""
    
    nodes = sorted(G.nodes())
    nid = {v: i for i, v in enumerate(nodes)}
    coloring = {}
    for v in nodes:
        vi = nid[v]
        color = sum((1 if sample.get(f"b_{vi}_{k}", 0) else 0) << k for k in range(L))
        coloring[v] = color
    return coloring

In [ ]:
###############################################################################
# EFFECTIVE FIELD COMPUTATION
###############################################################################

def compute_single_qubit_effective_field_iterative(
    couplings: List[float],
    bias: float,
) -> float:
    """
    O(N) iterative effective field computation.

    For QUBO (binary) variables b_i in {0,1}:
        T_i = sum_j J_{ij}
        f = |2h_i + T_i|
        for each J_ij: f = 0.5 * (|f + J_ij| + |f - J_ij|)  [= max(f, |J_ij|)]
        |F_i| = 0.5 * f

    This is equivalent to:
        |F_i| = 0.5 * max(|2h_i + T_i|, |J_1|, ..., |J_{N_i}|)

    Parameters
    ----------
    couplings : list of float
        Quadratic coefficients J_{ij} from the BQM involving qubit i.
    bias : float
        Linear coefficient h_i of qubit i in the BQM.

    Returns
    -------
    float
        The (approximate) effective field |F_i|.
    """
    T_i = sum(couplings)
    f = abs(2.0 * bias + T_i)
    for J_ij in couplings:
        f = 0.5 * (abs(f + J_ij) + abs(f - J_ij))
    return 0.5 * f


def compute_single_qubit_effective_field_exact(
    couplings: List[float],
    bias: float,
) -> float:
    """
    Exact effective field via O(2^{N_i}) recursive computation.

    Implements the recurrence relation:

        F(0, h') = |h'|
        F(n+1, h') = 0.5 * (F(n, h' + J_{n+1}) + F(n, h' - J_{n+1}))

    With initial condition h' = 2h_i + T_i (from spin-to-binary transform),
    and final result |F_i| = 0.5 * F(N_i, h').

    Complexity: O(2^{N_i}). Practical for max degree <= ~25.

    Parameters
    ----------
    couplings : list of float
        Quadratic coefficients J_{ij} from the BQM involving qubit i.
    bias : float
        Linear coefficient h_i of qubit i in the BQM.

    Returns
    -------
    float
        The exact mean effective field |F_i|.
    """
    N = len(couplings)

    if N == 0:
        return abs(bias)

    # For the binary->spin transform: h'_init = 2*h_i + T_i, where T_i = sum J_ij
    # Then use the spin recurrence, and multiply by 0.5 at the end.
    T_i = sum(couplings)
    h_init = 2.0 * bias + T_i

    def _recurse(n: int, h: float) -> float:
        """F(n, h) using couplings[0..n-1]."""
        if n == 0:
            return abs(h)
        J = couplings[n - 1]
        return 0.5 * (_recurse(n - 1, h + J) + _recurse(n - 1, h - J))

    return 0.5 * _recurse(N, h_init)


def compute_single_qubit_effective_field(
    couplings: List[float],
    bias: float,
    method: Literal['iterative', 'exact'] = 'iterative',
) -> float:
    """
    Compute the mean effective field |F_i| for a single QUBO qubit.

    Parameters
    ----------
    couplings : list of float
        Quadratic coefficients J_{ij} from the BQM involving qubit i.
    bias : float
        Linear coefficient h_i of qubit i in the BQM.
    method : 'iterative' or 'exact'
        'iterative': O(N) Algorithm.
        'exact': O(2^N) recursive computation.

    Returns
    -------
    float
        The effective field |F_i|.
    """
    if method == 'exact':
        return compute_single_qubit_effective_field_exact(couplings, bias)
    else:
        return compute_single_qubit_effective_field_iterative(couplings, bias)


def compute_all_effective_fields(
    bqm: BinaryQuadraticModel,
    method: Literal['iterative', 'exact'] = 'iterative',
    exact_degree_threshold: int = 25,
) -> Dict[str, float]:
    """
    Compute mean effective fields for all qubits in a BQM.

    Parameters
    ----------
    bqm : BinaryQuadraticModel
        The QUBO problem.
    method : 'iterative' or 'exact'
        'iterative': O(N) per qubit. Default.
        'exact': O(2^N) per qubit. Falls back to 'iterative'
                 for qubits with degree > exact_degree_threshold.
    exact_degree_threshold : int
        Maximum degree for exact computation. Qubits with more neighbors
        than this use the iterative method even when method='exact'.
        Default 25 (2^25 ~ 34M evaluations).

    Returns
    -------
    dict
        {variable_name: |F_i|} for every variable in the BQM.
    """
    effective_fields = {}

    for var in bqm.variables:
        bias = bqm.get_linear(var)
        couplings = [J for _, J in bqm.iter_neighborhood(var)]

        if method == 'exact' and len(couplings) > exact_degree_threshold:
            effective_fields[var] = compute_single_qubit_effective_field_iterative(
                couplings, bias
            )
        else:
            effective_fields[var] = compute_single_qubit_effective_field(
                couplings, bias, method=method
            )

    return effective_fields


In [ ]:
###############################################################################
# ANNEALING OFFSET COMPUTATION
###############################################################################

def compute_annealing_offsets(
    bqm: BinaryQuadraticModel,
    delta_max: float = 0.1,
    method: Literal['iterative', 'exact'] = 'iterative',
) -> Dict[str, float]:
    """
    Compute per-qubit annealing offsets for inhomogeneous driving.

    The offset for qubit i is:
        delta_i = |delta_max| * (1 - 2 * r_i)

    where r_i = |F_i| / max_k |F_k| in [0, 1].

    Strongly coupled qubits (high r_i) -> negative offset -> delayed schedule.
    Weakly coupled qubits (low r_i) -> positive offset -> advanced schedule.

    Parameters
    ----------
    bqm : BinaryQuadraticModel
        The QUBO to compute offsets for.
    delta_max : float
        Maximum offset magnitude. Default 0.1 (per manuscript). D-Wave
        Advantage supports up to ~0.5.
    method : 'iterative' or 'exact'
        Effective field computation method. Default 'iterative'.

    Returns
    -------
    dict
        {variable_name: delta_i} with values in [-|delta_max|, +|delta_max|].
    """
    delta_max = abs(delta_max)
    eff_fields = compute_all_effective_fields(bqm, method=method)

    if not eff_fields:
        return {}

    max_field = max(eff_fields.values())
    if max_field < 1e-15:
        return {var: 0.0 for var in eff_fields}

    r = {var: field / max_field for var, field in eff_fields.items()}
    offsets = {var: delta_max * (1.0 - 2.0 * r_i) for var, r_i in r.items()}

    return offsets

In [ ]:
###############################################################################
# LOGICAL -> PHYSICAL OFFSET MAPPING VIA EMBEDDING
###############################################################################

def map_offsets_to_physical(
    logical_offsets: Dict,
    embedding: Dict,
    num_physical_qubits: int,
) -> List[float]:
    """
    Map logical qubit annealing offsets to physical qubit offsets.

    All physical qubits in a chain receive the same offset as their
    logical qubit. Unused physical qubits get offset 0.

    Parameters
    ----------
    logical_offsets : dict
        {logical_variable: delta_i} from compute_annealing_offsets().
    embedding : dict
        {logical_variable: [physical_qubit_1, ...]}.
    num_physical_qubits : int
        Total physical qubits on the QPU.

    Returns
    -------
    list of float
        Length num_physical_qubits, indexed by physical qubit ID.
    """
    physical_offsets = [0.0] * num_physical_qubits

    for logical_var, physical_chain in embedding.items():
        offset = logical_offsets.get(logical_var, 0.0)
        for phys_q in physical_chain:
            if phys_q < num_physical_qubits:
                physical_offsets[phys_q] = offset

    return physical_offsets

In [ ]:
###############################################################################
# D-WAVE SAMPLING WITH INHOMOGENEOUS DRIVING
###############################################################################

def sample_with_inhomogeneous_driving(
    bqm: BinaryQuadraticModel,
    sampler: "DWaveSampler",
    delta_max: float = 0.1,
    num_reads: int = 1000,
    annealing_time: float = 20.0,
    embedding: Optional[Dict] = None,
    chain_strength: Optional[float] = None,
    greedy_postprocess: bool = False,
    effective_field_method: Literal['iterative', 'exact'] = 'iterative',
    label: str = "",
) -> Tuple["dimod.SampleSet", Dict]:
    """
    Submit a BQM to D-Wave with per-qubit annealing offsets.

    Pipeline:
      1. Compute mean effective fields for all logical qubits.
      2. Derive annealing offsets delta_i = |delta_max|*(1 - 2*r_i).
      3. Find or use a minor embedding.
      4. Map logical offsets to physical qubits via the embedding.
      5. Split reads across multiple submissions if necessary.
      6. Optionally apply greedy descent post-processing.

    Parameters
    ----------
    bqm : BinaryQuadraticModel
        The QUBO problem.
    sampler : DWaveSampler
        Raw D-Wave QPU sampler (not composite-wrapped).
    delta_max : float
        Maximum annealing offset magnitude. Default 0.1.
    num_reads : int
        Total annealing reads. Default 1000.
    annealing_time : float
        Annealing time in us. Default 20.
    embedding : dict or None
        Pre-computed embedding. If None, found automatically.
    chain_strength : float or None
        Chain strength. If None, uses D-Wave default.
    greedy_postprocess : bool
        Refine samples with SteepestDescentSolver. Default True.
    effective_field_method : 'iterative' or 'exact'
        How to compute effective fields. Default 'iterative'.
    label : str
        Optional D-Wave submission label.

    Returns
    -------
    (sample_set, info) : tuple
        sample_set: dimod.SampleSet (post-greedy if enabled).
        info: dict with effective fields, offsets, embedding, timing, etc.
    """
    if not DWAVE_AVAILABLE:
        raise ImportError(
            "dwave-system required. Install via: pip install dwave-ocean-sdk"
        )

    # -- Step 1: Compute logical annealing offsets --------------------------
    t_offset_start = time.perf_counter()

    eff_fields = compute_all_effective_fields(bqm, method=effective_field_method)
    logical_offsets = compute_annealing_offsets(
        bqm, delta_max=delta_max, method=effective_field_method
    )

    t_offset_end = time.perf_counter()
    offset_time_us = (t_offset_end - t_offset_start) * 1_000_000

    # -- Step 2: Find or validate embedding ---------------------------------
    if embedding is None:
        source_edgelist = list(bqm.quadratic.keys())
        target_edgelist = sampler.structure.edgelist
        embedding = minorminer.find_embedding(source_edgelist, target_edgelist)
        if not embedding:
            raise RuntimeError(
                f"Failed to embed BQM ({bqm.num_variables} vars) onto QPU."
            )

    # -- Step 3: Map offsets to physical qubits -----------------------------
    num_physical_qubits = sampler.properties['num_qubits']
    physical_offsets = map_offsets_to_physical(
        logical_offsets, embedding, num_physical_qubits
    )

    # -- Step 5: Submit with FixedEmbeddingComposite ------------------------
    fec = FixedEmbeddingComposite(sampler, embedding)
    results = None

    sampling_kwargs = dict(
        annealing_time=annealing_time,
        anneal_offsets=physical_offsets,
    )
    if chain_strength is not None:
        sampling_kwargs['chain_strength'] = chain_strength
    if label:
        sampling_kwargs['label'] = label

    for run_idx in range(1):
        cur_reads = num_reads
        cur_results = fec.sample(bqm, num_reads=cur_reads, **sampling_kwargs)
        if results is None:
            results = cur_results
        else:
            results = dimod.concatenate((results, cur_results))

    # -- Step 6: Chain break stats ------------------------------------------
    df = results.to_pandas_dataframe()
    mean_cbf = (
        df['chain_break_fraction'].mean()
        if 'chain_break_fraction' in df.columns
        else 0.0
    )
    num_phys_used = sum(len(chain) for chain in embedding.values())

    # -- Step 7: Optional greedy post-processing ----------------------------
    greedy_time_us = 0.0
    if greedy_postprocess:
        try:
            from dwave.samplers import SteepestDescentSolver

            t_greedy_start = time.perf_counter()

            nonvars = {
                'energy', 'num_occurrences', 'chain_break_fraction', 'num_steps'
            }
            var_cols = [c for c in df.columns if c not in nonvars]

            expanded = []
            for _, row in df.iterrows():
                sample = {v: row[v] for v in var_cols}
                expanded.extend([sample] * int(row.get('num_occurrences', 1)))

            energies = [bqm.energy(s) for s in expanded]
            initial_ss = dimod.SampleSet.from_samples(
                samples_like=expanded,
                energy=energies,
                vartype=bqm.vartype,
            )

            results = SteepestDescentSolver().sample(bqm, initial_states=initial_ss)

            greedy_time_us = (time.perf_counter() - t_greedy_start) * 1_000_000

        except ImportError:
            warnings.warn("SteepestDescentSolver unavailable; skipping greedy.")

    # -- Build info ---------------------------------------------------------
    info = {
        'logical_offsets': logical_offsets,
        'effective_fields': eff_fields,
        'embedding': embedding,
        'num_physical_qubits_used': num_phys_used,
        'chain_break_fraction': mean_cbf,
        'greedy_time_us': greedy_time_us,
        'offset_computation_time_us': offset_time_us,
        'effective_field_method': effective_field_method,
        'delta_max': delta_max,
        'num_reads': num_reads,
    }
    return results, info

In [ ]:
###############################################################################
# VALIDATION & METRICS
###############################################################################

def validate_coloring(G: nx.Graph, coloring: dict) -> bool:
    if coloring is None or len(coloring) != G.number_of_nodes():
        return False
    return all(coloring[u] != coloring[v] for u, v in G.edges())


def num_colors_used(coloring: dict) -> int:
    return len(set(coloring.values())) if coloring else float("inf")


def compute_tts(sample_set, best_energy, anneal_us=20.0, readout_us=100.0, p_target=0.5):
    """
    TTS = (t_anneal + t_readout) · ⌈ log(1-p_target) / log(1-p_success) ⌉
    """
    energies = sample_set.record.energy
    occ = sample_set.record.num_occurrences
    total = occ.sum()
    hits = occ[np.abs(energies - best_energy) < 1e-6].sum()
    if hits == 0:
        return float("inf")
    p = hits / total
    if p >= 1.0:
        return anneal_us + readout_us
    r = math.ceil(math.log(1.0 - p_target) / math.log(1.0 - p))
    return (anneal_us + readout_us) * r

In [ ]:
###############################################################################
# SAMPLING INTERFACE
###############################################################################

def sample_dwave(
    bqm: BinaryQuadraticModel,
    sampler: "DWaveSampler",
    num_reads: int = 1000,
    annealing_time: float = 20.0,
    use_inhomogeneous_driving: bool = False,
    delta_max: float = 0.1,
    effective_field_method: Literal['iterative', 'exact'] = 'iterative',
    embedding: Optional[Dict] = None,
    chain_strength: Optional[float] = None,
    greedy_postprocess: bool = False,
    label: str = "",
) -> Tuple["dimod.SampleSet", Dict]:
    """
    Unified D-Wave sampling with or without inhomogeneous driving.

    When use_inhomogeneous_driving is False, samples via standard
    EmbeddingComposite (or FixedEmbeddingComposite if embedding provided).

    Returns (sample_set, info_dict) in both cases, with matching structure.
    """
    if use_inhomogeneous_driving:
        return sample_with_inhomogeneous_driving(
            bqm=bqm,
            sampler=sampler,
            delta_max=delta_max,
            num_reads=num_reads,
            annealing_time=annealing_time,
            embedding=embedding,
            chain_strength=chain_strength,
            greedy_postprocess=greedy_postprocess,
            effective_field_method=effective_field_method,
            label=label,
        )

    # -- Standard (homogeneous) sampling ------------------------------------
    if not DWAVE_AVAILABLE:
        raise ImportError("dwave-system required for hardware sampling.")

    if embedding is not None:
        composite = FixedEmbeddingComposite(sampler, embedding)
    else:
        composite = EmbeddingComposite(sampler)

    sampling_kwargs = dict(
        num_reads=num_reads,
        annealing_time=annealing_time,
    )
    if chain_strength is not None:
        sampling_kwargs['chain_strength'] = chain_strength
    if label:
        sampling_kwargs['label'] = label

    results = composite.sample(bqm, **sampling_kwargs)

    df = results.to_pandas_dataframe()
    mean_cbf = (
        df['chain_break_fraction'].mean()
        if 'chain_break_fraction' in df.columns
        else 0.0
    )

    # Greedy post-processing
    greedy_time_us = 0.0
    if greedy_postprocess:
        try:
            from dwave.samplers import SteepestDescentSolver

            t_start = time.perf_counter()

            nonvars = {
                'energy', 'num_occurrences', 'chain_break_fraction', 'num_steps'
            }
            var_cols = [c for c in df.columns if c not in nonvars]

            expanded = []
            for _, row in df.iterrows():
                sample = {v: row[v] for v in var_cols}
                expanded.extend([sample] * int(row.get('num_occurrences', 1)))

            energies = [bqm.energy(s) for s in expanded]
            initial_ss = dimod.SampleSet.from_samples(
                samples_like=expanded,
                energy=energies,
                vartype=bqm.vartype,
            )

            results = SteepestDescentSolver().sample(bqm, initial_states=initial_ss)
            greedy_time_us = (time.perf_counter() - t_start) * 1_000_000

        except ImportError:
            pass

    actual_embedding = embedding
    if actual_embedding is None and hasattr(composite, 'embedding'):
        actual_embedding = composite.embedding

    info = {
        'logical_offsets': {},
        'effective_fields': {},
        'embedding': actual_embedding,
        'num_physical_qubits_used': (
            sum(len(c) for c in actual_embedding.values())
            if actual_embedding else 0
        ),
        'chain_break_fraction': mean_cbf,
        'greedy_time_us': greedy_time_us,
        'offset_computation_time_us': 0.0,
        'effective_field_method': 'none',
        'delta_max': 0.0,
        'num_runs': 1,
        'reads_per_run': num_reads,
    }
    return results, info

In [ ]:
###############################################################################
# DIAGNOSTIC UTILITIES
###############################################################################

def summarize_offsets(offsets: Dict[str, float]) -> Dict[str, float]:
    """Summary statistics of annealing offsets."""
    if not offsets:
        return {}
    vals = list(offsets.values())
    return {
        'num_qubits': len(vals),
        'min_offset': min(vals),
        'max_offset': max(vals),
        'mean_offset': float(np.mean(vals)),
        'std_offset': float(np.std(vals)),
        'num_delayed': sum(1 for v in vals if v < -1e-15),
        'num_advanced': sum(1 for v in vals if v > 1e-15),
        'num_neutral': sum(1 for v in vals if abs(v) <= 1e-15),
    }


def summarize_effective_fields(fields: Dict[str, float]) -> Dict[str, float]:
    """Summary statistics of effective fields."""
    if not fields:
        return {}
    vals = list(fields.values())
    return {
        'num_qubits': len(vals),
        'min_field': min(vals),
        'max_field': max(vals),
        'mean_field': float(np.mean(vals)),
        'std_field': float(np.std(vals)),
        'median_field': float(np.median(vals)),
    }


In [ ]:
###############################################################################
# SOLVER INTERFACE
###############################################################################

def get_sampler(use_dwave=False, token=None, solver=None,
                use_inhomogeneous_driving=False):
    if use_dwave and DWAVE_AVAILABLE:
        try:
            kw = {}
            if token:
                kw["token"] = token
            if solver:
                kw["solver"] = solver
            raw = DWaveSampler(**kw)
            if use_inhomogeneous_driving:
                # Return raw sampler — sample_dwave() handles
                # embedding + offsets internally
                return raw, "dwave_ihd"
            else:
                return EmbeddingComposite(raw), "dwave"
        except Exception as e:
            print(f"  [WARN] D-Wave unavailable ({e}); falling back to SA.")
    return SimulatedAnnealingSampler(), "simulated"


def solve_bqm(bqm, sampler, stype, num_reads=5000, anneal_us=20.0,
              use_inhomogeneous_driving=False, delta_max=0.1,
              effective_field_method='iterative',
              greedy_postprocess=False):
    """
    Solve a BQM, routing to the appropriate backend.

    For stype == "dwave_ihd": uses sample_dwave() with inhomogeneous driving.
    For stype == "dwave":     uses EmbeddingComposite (already wrapped).
    For stype == "simulated": uses SimulatedAnnealingSampler.
    """
    t0 = time.time()

    if stype == "dwave_ihd":
        # --- Inhomogeneous driving path ---
        # sampler is a raw DWaveSampler; sample_dwave handles embedding
        ss, ihd_info = sample_dwave(
            bqm,
            sampler=sampler,
            num_reads=num_reads,
            annealing_time=anneal_us,
            use_inhomogeneous_driving=True,
            delta_max=delta_max,
            effective_field_method=effective_field_method,
            greedy_postprocess=greedy_postprocess,
        )
        wall = time.time() - t0

        emb = ihd_info.get('embedding', {})
        emb_info = {
            "wall_time_s": wall,
            "num_physical_qubits": ihd_info.get('num_physical_qubits_used'),
            "max_chain_length": max((len(c) for c in emb.values()), default=0) if emb else 0,
            "chain_break_fraction": ihd_info.get('chain_break_fraction', 0.0),
            "ihd_info": ihd_info,  # full IHD diagnostics
        }
        return ss, emb_info

    # --- Standard D-Wave or simulated path ---
    kw = {"num_reads": num_reads}
    if stype == "dwave":
        kw["annealing_time"] = anneal_us
        kw["return_embedding"] = True

    ss = sampler.sample(bqm, **kw)
    wall = time.time() - t0

    emb_info = {"wall_time_s": wall}
    if stype == "dwave":
        try:
            emb = ss.info.get("embedding_context", {}).get("embedding", {})
            emb_info["num_physical_qubits"] = sum(len(c) for c in emb.values())
            emb_info["max_chain_length"] = max((len(c) for c in emb.values()), default=0)
        except Exception:
            pass

    return ss, emb_info


def estimate_physical_qubits(bqm):
    
    """Estimate physical qubits on Zephyr Z20 via minorminer."""
    
    if not DWAVE_AVAILABLE:
        return None
    try:
        target = dnx.zephyr_graph(20)
        source = dimod.to_networkx_graph(bqm)
        emb = minorminer.find_embedding(source, target, timeout=30)
        return sum(len(c) for c in emb.values()) if emb else None
    except Exception:
        return None


In [ ]:
###############################################################################
# BENCHMARK ONE GRAPH
###############################################################################

def benchmark_graph(G, name, sampler, stype, num_reads=5000,
                    anneal_us=20.0, est_embed=False,
                    use_inhomogeneous_driving=False,
                    delta_max=0.1,
                    effective_field_method='iterative',
                    greedy_postprocess=False):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    degs = [d for _, d in G.degree()]
    avg_d = float(np.mean(degs)) if degs else 0.0
    max_d = max(degs) if degs else 0

    t0 = time.time()
    C = brooks_bound(G)
    brooks_t = time.time() - t0

    ihd_tag = " +IHD" if use_inhomogeneous_driving else ""
    print(f"\n{'═'*72}")
    print(f"  {name}   |V|={n}  |E|={m}  Δ={max_d}  Brooks≤{C}{ihd_tag}")
    print(f"{'═'*72}")

    res = dict(graph_name=name, num_nodes=n, num_edges=m,
               avg_degree=round(avg_d, 2), max_degree=max_d,
               brooks_bound=C, brooks_time_s=round(brooks_t, 5),
               inhomogeneous_driving=use_inhomogeneous_driving)

    # Common kwargs for solve_bqm
    solve_kw = dict(
        num_reads=num_reads,
        anneal_us=anneal_us,
        use_inhomogeneous_driving=use_inhomogeneous_driving,
        delta_max=delta_max,
        effective_field_method=effective_field_method,
        greedy_postprocess=greedy_postprocess,
    )

    # ── ONE-HOT ──────────────────────────────────────────────────────────
    print(f"  [OH]  Building QUBO  (C={C}, vars={C*(n+1)}) ...")
    t0 = time.time()
    try:
        bqm_oh, ioh = build_onehot_qubo(G, C)
        bt = time.time() - t0
        print(f"  [OH]  Built in {bt:.3f}s   ({bqm_oh.num_variables} vars, "
              f"{bqm_oh.num_interactions} couplers)")

        ss_oh, ei_oh = solve_bqm(bqm_oh, sampler, stype, **solve_kw)

        best_nc, best_en, best_cnt, n_valid = float("inf"), float("inf"), 0, 0
        for s, e, o in ss_oh.data(["sample", "energy", "num_occurrences"]):
            col = decode_onehot(dict(s), G, C)
            if col and validate_coloring(G, col):
                nc = num_colors_used(col)
                n_valid += o
                if nc < best_nc or (nc == best_nc and e < best_en):
                    best_nc, best_en, best_cnt = nc, e, o
                elif nc == best_nc and abs(e - best_en) < 1e-6:
                    best_cnt += o

        tts = compute_tts(ss_oh, best_en, anneal_us) if best_en < float("inf") else float("inf")

        phys = ei_oh.get("num_physical_qubits")
        if phys is None and est_embed:
            print("  [OH]  Estimating embedding ...")
            phys = estimate_physical_qubits(bqm_oh)

        # IHD diagnostics
        ihd_info_oh = ei_oh.get("ihd_info", {})
        if ihd_info_oh:
            off_summary = summarize_offsets(ihd_info_oh.get('logical_offsets', {}))
            print(f"  [OH]  IHD: {off_summary.get('num_delayed',0)} delayed, "
                  f"{off_summary.get('num_advanced',0)} advanced  "
                  f"CBF={ei_oh.get('chain_break_fraction', 0):.3f}")

        print(f"  [OH]  Best: {best_nc} colors  ({best_cnt}/{num_reads})  "
              f"valid={n_valid}/{num_reads}")

        res.update(oh_num_logical=ioh["num_logical_qubits"],
                   oh_num_physical=phys, oh_build_s=round(bt, 4),
                   oh_solve_s=round(ei_oh["wall_time_s"], 4),
                   oh_best_ncolors=best_nc if best_nc < float("inf") else None,
                   oh_best_count=best_cnt, oh_valid=n_valid,
                   oh_tts_us=tts if tts < float("inf") else None,
                   oh_cbf=ei_oh.get("chain_break_fraction"))

    except Exception as exc:
        print(f"  [OH]  FAILED: {exc}")
        import traceback; traceback.print_exc()
        res.update(oh_num_logical=None, oh_num_physical=None,
                   oh_build_s=None, oh_solve_s=None,
                   oh_best_ncolors=None, oh_best_count=0,
                   oh_valid=0, oh_tts_us=None, oh_cbf=None)

    # ── BINARY ───────────────────────────────────────────────────────────
    L = max(1, math.ceil(math.log2(C))) if C > 1 else 1
    print(f"  [BIN] Building HUBO  (C={C}, L={L}, pre-quad vars={n*L}) ...")
    t0 = time.time()
    try:
        hubo, ibin = build_binary_hubo(G, C)
        ht = time.time() - t0
        print(f"  [BIN] HUBO built in {ht:.3f}s   "
              f"({ibin['hubo_num_terms']} terms, max deg {ibin['hubo_max_degree']})")

        print(f"  [BIN] Quadratizing ...")
        t0 = time.time()
        bqm_bin, ibin = quadratize_binary(hubo, ibin, G)
        qt = time.time() - t0
        tot_bt = ht + qt
        print(f"  [BIN] QUBO: {bqm_bin.num_variables} vars "
              f"(+{ibin['num_auxiliary']} aux, "
              f"theory {ibin['theoretical_auxiliary']})  "
              f"{bqm_bin.num_interactions} couplers   "
              f"quad {qt:.3f}s")

        ss_bin, ei_bin = solve_bqm(bqm_bin, sampler, stype, **solve_kw)

        best_nc, best_en, best_cnt, n_valid = float("inf"), float("inf"), 0, 0
        for s, e, o in ss_bin.data(["sample", "energy", "num_occurrences"]):
            col = decode_binary(dict(s), G, L)
            if validate_coloring(G, col):
                nc = num_colors_used(col)
                n_valid += o
                if nc < best_nc or (nc == best_nc and e < best_en):
                    best_nc, best_en, best_cnt = nc, e, o
                elif nc == best_nc and abs(e - best_en) < 1e-6:
                    best_cnt += o

        tts = compute_tts(ss_bin, best_en, anneal_us) if best_en < float("inf") else float("inf")

        phys = ei_bin.get("num_physical_qubits")
        if phys is None and est_embed:
            print("  [BIN] Estimating embedding ...")
            phys = estimate_physical_qubits(bqm_bin)

        # IHD diagnostics
        ihd_info_bin = ei_bin.get("ihd_info", {})
        if ihd_info_bin:
            off_summary = summarize_offsets(ihd_info_bin.get('logical_offsets', {}))
            print(f"  [BIN] IHD: {off_summary.get('num_delayed',0)} delayed, "
                  f"{off_summary.get('num_advanced',0)} advanced  "
                  f"CBF={ei_bin.get('chain_break_fraction', 0):.3f}")

        print(f"  [BIN] Best: {best_nc} colors  ({best_cnt}/{num_reads})  "
              f"valid={n_valid}/{num_reads}")

        res.update(bin_L=L,
                   bin_pre_quad=ibin["num_logical_qubits_pre_quad"],
                   bin_post_quad=ibin["num_logical_qubits_post_quad"],
                   bin_aux=ibin["num_auxiliary"],
                   bin_aux_theory=ibin["theoretical_auxiliary"],
                   bin_num_physical=phys,
                   bin_build_s=round(tot_bt, 4),
                   bin_solve_s=round(ei_bin["wall_time_s"], 4),
                   bin_best_ncolors=best_nc if best_nc < float("inf") else None,
                   bin_best_count=best_cnt, bin_valid=n_valid,
                   bin_tts_us=tts if tts < float("inf") else None,
                   bin_hubo_deg=ibin["hubo_max_degree"],
                   bin_cbf=ei_bin.get("chain_break_fraction"))

    except Exception as exc:
        print(f"  [BIN] FAILED: {exc}")
        import traceback; traceback.print_exc()
        res.update(bin_L=L, bin_pre_quad=None, bin_post_quad=None,
                   bin_aux=None, bin_aux_theory=None, bin_num_physical=None,
                   bin_build_s=None, bin_solve_s=None,
                   bin_best_ncolors=None, bin_best_count=0,
                   bin_valid=0, bin_tts_us=None, bin_hubo_deg=None,
                   bin_cbf=None)

    return res


In [ ]:
# options:

generate_test_graphs = False
graph_dir            = "<INPUT GRAPH DIRECTORY HERE>"
output_dir           = "<INPUT OUTPUT DIRECTORY HERE>"
use_dwave            = True
dwave_token          = "<INPUT D-WAVE TOKEN HERE>"
dwave_solver         = "Advantage2_system1"
num_reads            = 1000
anneal_time          = 20.0
estimate_embedding   = True
max_nodes            = 100

# ── Inhomogeneous driving options ──
use_inhomogeneous_driving = False        # Toggle IHD on/off
delta_max                 = 0.1          # Max offset magnitude
effective_field_method    = 'iterative'  # 'iterative' (O(N)) or 'exact' (O(2^N))
greedy_postprocess        = False        # SteepestDescentSolver post-processing

# Optional: real D-Wave hardware
try:
    from dwave.system import DWaveSampler, EmbeddingComposite, FixedEmbeddingComposite
    import minorminer
    import dwave_networkx as dnx
    DWAVE_AVAILABLE = True
except ImportError:
    DWAVE_AVAILABLE = False

# main loop:

os.makedirs(output_dir, exist_ok=True)

# Load / generate graphs
if generate_test_graphs:
    gdir = os.path.join(output_dir, "test_graphs")
    graphs = generate_test_graphs(gdir)
elif graph_dir:
    pkls = sorted(glob.glob(os.path.join(graph_dir, "*.pkl")))
    if not pkls:
        sys.exit(f"No .pkl files in {graph_dir}")
    graphs = []
    for p in pkls:
        basename = os.path.splitext(os.path.basename(p))[0]
        with open(p, "rb") as f:
            obj = pickle.load(f)
        # Handle both single graphs and lists/tuples of graphs
        if isinstance(obj, nx.Graph):
            graphs.append((basename, obj))
        elif isinstance(obj, (list, tuple)):
            for i, item in enumerate(obj):
                if isinstance(item, nx.Graph):
                    graphs.append((f"{basename}_{i}", item))
                else:
                    print(f"  [WARN] {basename}[{i}]: unexpected type "
                          f"{type(item).__name__}, skipping")
        else:
            print(f"  [WARN] {basename}: unexpected type "
                  f"{type(obj).__name__}, skipping")
    print(f"Loaded {len(graphs)} graphs from {graph_dir}")
else:
    sys.exit("Specify --graph-dir or --generate-test-graphs.  Use -h for help.")

orig = len(graphs)
graphs = [(n, G) for n, G in graphs if G.number_of_nodes() <= max_nodes]
if len(graphs) < orig:
    print(f"Filtered to {len(graphs)} graphs (≤{max_nodes} nodes)")

tok = dwave_token or os.environ.get("DWAVE_API_TOKEN")
sampler, stype = get_sampler(
    use_dwave, tok,
    solver=dwave_solver,
    use_inhomogeneous_driving=use_inhomogeneous_driving,
)
print(f"Sampler: {stype}"
      + (f"  (IHD δ_max={delta_max}, method={effective_field_method})"
         if stype == "dwave_ihd" else ""))

results = []
for name, G in graphs:
    try:
        r = benchmark_graph(
            G, name, sampler, stype,
            num_reads, anneal_time,
            estimate_embedding,
            use_inhomogeneous_driving=use_inhomogeneous_driving,
            delta_max=delta_max,
            effective_field_method=effective_field_method,
            greedy_postprocess=greedy_postprocess,
        )
        results.append(r)
    except Exception as exc:
        print(f"  [ERR] {name}: {exc}")
        import traceback; traceback.print_exc()

if not results:
    sys.exit("No results.")

df = pd.DataFrame(results)
csv = os.path.join(output_dir, "execution_results.csv")
df.to_csv(csv, index=False)
print(f"\nResults → {csv}")

# Summary table
print(f"\n{'═'*100}")
show = [c for c in [
    "graph_name","num_nodes","num_edges","max_degree","brooks_bound",
    "oh_num_logical","dw_num_logical","bin_pre_quad","bin_post_quad",
    "oh_best_ncolors","dw_best_ncolors","bin_best_ncolors",
    "oh_tts_us","dw_tts_us","bin_tts_us",
    "oh_cbf","dw_cbf","bin_cbf",
] if c in df.columns]
print(df[show].to_string(index=False))

try:
    generate_plots(df, os.path.join(output_dir, "plots"))
except ImportError:
    print("\n  [WARN] matplotlib not installed — skipping plots.")
    print("         Install with: pip install matplotlib")
print("\nDone.")
